# nb_06_reset_clean — wipe pipeline state for a clean E2E run

**DESTRUCTIVE.** Resets the pipeline to a blank slate so the Sprint 10 E2E test starts from
zero: it **empties every pipeline Delta table** (`file_metadata`, `ingestion_state`,
`ingestion_log`, `skipped_log`, `run_progress`, `throttle_log`) **and deletes every document
from the Azure AI Search index** (the index *schema* is preserved — only the docs are removed).
The `config` table is **left untouched** (endpoints/settings survive).

## Inputs
- Attached lakehouse `aws_connect_lh` (so `spark.table(...)` resolves).
- `config` keys: `kv_name`, `search_key_secret`, `search_endpoint`, `search_index_name`.
- Key Vault **secret get** on the Search admin key (same as nb_03/nb_05).

## Outputs / effects
- All six pipeline Delta tables emptied (rows deleted; tables + schema kept).
- AI Search index emptied (0 documents; schema kept).
- Prints before/after counts for every table and the index.

## How to run
- Interactive: set `CONFIRM = True` in the guard cell, then Run All.
- Headless (`run_fabric_nb.ps1`): pass the `CONFIRM` parameter as `true`, or leave the default
  in place. The guard prevents an accidental wipe.

> Does **not** touch S3 or the `config` table. To rebuild the index schema from scratch use
> `nb_02`; this notebook only clears documents.


## Safety guard
This notebook refuses to run unless `CONFIRM` is `True`. When run as a parameterized job, the
`CONFIRM` parameter (a Fabric *parameter cell* — the tag below) overrides this value.


In [ ]:
CONFIRM = False  # set True (or pass CONFIRM=true as a job parameter) to actually wipe


In [ ]:
if not CONFIRM:
    raise SystemExit('Refusing to wipe: set CONFIRM = True (or pass CONFIRM=true) to proceed.')
print('CONFIRM is set — proceeding with reset.')


## Load config


In [ ]:
import json, traceback, notebookutils
_TRACE = []
def _step(label, fn):
    try:
        out = fn()
        _TRACE.append({'step': label, 'ok': True})
        return out
    except Exception as e:
        _TRACE.append({'step': label, 'ok': False,
                       'error': f'{type(e).__name__}: {e}',
                       'traceback': traceback.format_exc()})
        try:
            notebookutils.fs.put('Files/_diag/nb06_trace.json', json.dumps(_TRACE, indent=1), True)
        except Exception:
            pass
        raise

def _flush_trace():
    notebookutils.fs.put('Files/_diag/nb06_trace.json', json.dumps(_TRACE, indent=1), True)

cfg = _step('load_config', lambda: {r['key']: r['value'] for r in spark.table('config').collect()})
SEARCH_ENDPOINT = cfg['search_endpoint'].rstrip('/')
INDEX_NAME = cfg['search_index_name']
SEARCH_API = '2024-07-01'
_flush_trace()


## 1. Empty the pipeline Delta tables
Each table is emptied with `DELETE FROM` (keeps the table + schema, drops all rows). Tables that
don't exist yet (e.g. `run_progress` before the first nb_03 run) are skipped. Row counts are
printed before and after so the wipe is auditable.


In [ ]:
PIPELINE_TABLES = ['file_metadata', 'ingestion_state', 'ingestion_log',
                   'skipped_log', 'run_progress', 'throttle_log']

def _table_exists(name):
    try:
        spark.table(name); return True
    except Exception:
        return False

for t in PIPELINE_TABLES:
    if not _table_exists(t):
        print(f'  {t:16s}: (does not exist yet — skipped)')
        continue
    before = spark.table(t).count()
    _step(f'delete_{t}', lambda t=t: spark.sql(f'DELETE FROM {t}'))
    after = spark.table(t).count()
    print(f'  {t:16s}: {before} -> {after}')
_flush_trace()
print('Delta tables emptied.')


## 2. Delete every document from the AI Search index
Pages through the index 1000 chunk_ids at a time and issues batched `delete` actions until the
index is empty. The index **schema is preserved** — only documents are removed. Uses the Search
admin key from Key Vault (the Search token audience isn't issuable in Fabric).


In [ ]:
import requests, notebookutils, time

VAULT_URL = f"https://{cfg['kv_name']}.vault.azure.net/"
SEARCH_KEY = _step('get_search_key', lambda: notebookutils.credentials.getSecret(VAULT_URL, cfg['search_key_secret']))
HDRS = {'api-key': SEARCH_KEY, 'Content-Type': 'application/json'}

def _count():
    r = requests.get(f'{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/$count?api-version={SEARCH_API}',
                     headers={'api-key': SEARCH_KEY}, timeout=(10, 30))
    r.raise_for_status()
    return int(r.text.strip().lstrip('\ufeff'))

before = _step('search_count_before', _count)
print(f'Search index "{INDEX_NAME}" docs before: {before}')
search_url = f'{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version={SEARCH_API}'
index_url  = f'{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/index?api-version={SEARCH_API}'
deleted = 0
def _delete_all():
    global deleted
    while True:
        r = requests.post(search_url, headers=HDRS,
                          json={'search': '*', 'select': 'chunk_id', 'top': 1000}, timeout=(10, 60))
        r.raise_for_status()
        ids = [d['chunk_id'] for d in r.json().get('value', [])]
        if not ids:
            break
        actions = [{'@search.action': 'delete', 'chunk_id': i} for i in ids]
        dr = requests.post(index_url, headers=HDRS, json={'value': actions}, timeout=(10, 120))
        dr.raise_for_status()
        deleted += len(ids)
        time.sleep(1)  # let deletes commit before the next page query
_step('search_delete_all', _delete_all)

# Poll until the count settles at 0 (deletes are eventually consistent).
for _ in range(60):
    if _count() == 0:
        break
    time.sleep(2)
after = _step('search_count_after', _count)
_flush_trace()
print(f'deleted {deleted} docs; Search index docs after: {after}')
assert after == 0, f'index not empty after reset (still {after})'


## 3. Confirm clean slate


In [ ]:
print('=== RESET COMPLETE ===')
for t in PIPELINE_TABLES:
    n = spark.table(t).count() if _table_exists(t) else 0
    print(f'  {t:16s}: {n} rows')
print(f'  search index    : {_count()} docs')
print('Ready for a fresh E2E run (seed S3 -> nb_01 -> nb_03 -> nb_07).')
